# 🎯 [MNT10_M02] 이동정비반 보유 장비/공구 및 출동 가능 상태 정보 전처리 모듈 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT10/mnt10_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT10_M02 |
| **미션 주제** | 이동정비반 보유 장비/공구 및 출동 가능 상태 정보 전처리 모듈 구현 |
| **소요 시간** | 약 2.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
야전에서 장비가 고장 나면 이동정비반이 얼마나 빨리 도착해 고치느냐가 부대 전투력 회복 속도를 좌우하지만, 가까운 정비반부터 보내는 방식은 요청이 몰리는 지역의 정비반에 부하를 집중시키고 핵심 장비를 뒤로 미루는 문제가 있다. 본 프로젝트는 정비인력의 부하율과 공정 균형을 데이터로 예측·관리하는 것을 목표로, 교육용 가상 지역의 긴급 정비 요청과 이동정비반 자료를 코드로 만들어 분석한다. 요청 데이터 스키마 정의, 정비반 상태 자료 정제, 거리 기준 Baseline과 탐색적 분석, 중요도·정비 소요시간·이동 경로를 종합한 우선순위 엔진, 출동 지연 위험도와 현장 수리 가능성 점수, 관제 대시보드와 설명가능성, 정비대대장 승인(HITL) 시뮬레이터, 동시 다발 요청 스트레스 시험, AAR 가중치 재튜닝과 효과 산출, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 핵심 장비의 가동중단 시간을 줄이면서 정비반 간 부하를 고르게 하고, 출동 순서 변경과 재배치는 지휘관이 근거를 보고 결정하도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 산출물(deliverables) 디렉토리 구성 안내

본 과제는 실무 현장에서 바로 쓰일 수 있는 국방 소프트웨어 아키텍처 표준을 지향합니다.  
과제를 진행하며 생성되는 산출물은 아래 규칙에 따라 각 폴더에 체계적으로 저장해 주세요.

| 디렉토리 | 군수 실무 비유 | 주 역할 및 산출물 파일 | 분리 목적 |
| :--- | :--- | :--- | :--- |
| **`docs/`** | 야전 교범 / 정비 지침서 | `mnt10_logistics_domain_spec.md` (미션1), `mnt10_operations_guide.md` (미션10) | 코드를 모르는 지휘관/후임자도 데이터 규격과 비상 매뉴얼을 이해하도록 보장 |
| **`notebooks/`** | 정비 시험실 / 연구실 | `mnt10_eda_analysis.ipynb` (미션3), 본 실습 노트북 | AI와 대화하며 자유롭게 데이터를 시각화하고 알고리즘을 반복 실험 |
| **`src/`** | 실제 장착 핵심 부품 모듈 | `src/etl/`, `src/features/`, `src/models/`, `src/dashboard/`, `src/optimization/` | 운영 서버에서 자동 배치 실행 가능한 순수 파이썬 모듈(`.py`)로 부품화 |
| **`reports/`** | 지휘관 결심 보고서 | `mnt10_roi_analysis.md` (미션9), AAR 오배정 분석서 | 정비대대장/군수참모에게 정비 지연 15% 감축 등 정량 ROI 성과 보고 |
| **`pipeline/`** | 자동화 컨베이어 벨트 | `pipeline/mnt10_retrain.py` (미션10) | 긴급 요청 유입 시 전처리부터 관제까지 자동 구동 및 재학습 파이프라인 |

> **💡 [바이브 코딩(AI 협업) 실습 요령]**  
> 1. **노트북에서 먼저 검증**: 본 노트북의 각 미션 셀에서 AI에게 질의하며 단계별 코드를 실행하고 결과를 확인합니다.  
> 2. **AI에게 모듈화 요청**: 검증 완료 후 AI에게 *"이 코드를 `src/.../` 파일에 넣을 수 있게 함수/클래스로 모듈화해줘"*라고 요청합니다.  
> 3. **디렉토리별 저장**: 제안된 코드를 해당 파일에 저장하여 자연스럽게 실무 패키지를 완성합니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 이동정비반 보유 장비/공구 및 출동 가능 상태 정보 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

이동정비반 보유 장비 및 출동 가능 상태 정보 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **이동정비반의 보유 장비·공구·인력·출동 가능 상태 로그를 정제하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **중복 보고 제거, 정비반 ID·상태값·시각 표기 통일, 연료 단위 통일, 불가능한 인원·음수 공구 수량 결측 처리, 좌표 이상값 보간, KNN 결측 보정 등 정제 규칙 10종을 적용 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/etl/mnt10_data_cleansing.py 및 정제 품질 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 특수 예외(비계획 정비, 긴급 불출 등) 데이터 보존의 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 규칙 10종 적용, 데이터 정합성 검증 통과율 100%


- 진행 기준: ETL 파이프라인 구현 및 정제된 마스터 데이터셋 구축


- 검증 방법: 전후 데이터 통계량(기술통계) 비교 및 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""이동정비반의 보유 장비·공구·인력·출동 가능 상태 로그(표기 불일치·중복·단위 혼재·결측이 섞인 가상 원천 데이터)에 정제 규칙 10종을 적용하고, 긴급 불출 등 군수 예외 데이터는 보존하면서 정합성 검증 통과율과 정제 전후 통계를 확인한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

# 1) [교육용가상자료] 원천 상태 로그: 정비반 6개 x 30일 x 하루 2회 보고 + 일부러 넣은 오류
rng = np.random.default_rng(2)
rows = []
for d in range(30):
    for h in (8, 18):
        for t in range(1, 7):
            rows.append({"team": f"MT{t}", "ts": f"2026-03-{d + 1:02d} {h:02d}:00", "status": str(rng.choice(["ready", "moving", "repairing"], p=[0.6, 0.2, 0.2])),
                         "crew": int(rng.integers(3, 6)), "fuel": float(round(rng.uniform(35, 100), 1)), "fuel_unit": "%",
                         "diag_kit": int(rng.integers(1, 3)), "lift_kit": int(rng.integers(0, 2)), "x": float(round(rng.uniform(0, 60), 1)),
                         "y": float(round(rng.uniform(0, 60), 1)), "emergency_issue": False})
raw = pd.DataFrame(rows)
n0 = len(raw)
idx = rng.choice(n0, 150, replace=False)
a, b, c, d_, e, f, g, h_, k, m = np.split(idx, 10)
raw.loc[a, "team"] = raw.loc[a, "team"].str.replace("MT", "mt-")                            # ID 표기 불일치
raw.loc[b, "status"] = rng.choice(["READY", "ready ", "대기", "Repairing"], len(b))          # 상태값 표기 불일치
raw.loc[c, "ts"] = pd.to_datetime(raw.loc[c, "ts"]).dt.strftime("%Y%m%d %H%M")               # 시각 형식 혼재
raw.loc[d_, "fuel"] = (raw.loc[d_, "fuel"] * 4).round(0); raw.loc[d_, "fuel_unit"] = "L"    # 단위 혼재(가상 탱크 400L)
raw.loc[e, "crew"] = rng.choice([-1, 0, 25], len(e))                                        # 불가능 값
raw.loc[f, "diag_kit"] = -1                                                                # 음수 수량
raw.loc[g, "x"] = 999.0                                                                    # 좌표 범위 밖
raw.loc[h_, ["crew", "fuel"]] = np.nan                                                     # 결측
raw.loc[k, "lift_kit"] = 6; raw.loc[k, "emergency_issue"] = True                           # 긴급 불출(예외: 보존 대상)
raw.loc[m, "status"] = None                                                                # 상태 결측
raw = pd.concat([raw, raw.sample(12, random_state=1)], ignore_index=True)                  # 중복 보고
print(f"원천 로그 {len(raw)}건 (중복 12건, 오류 주입 {len(idx)}건 포함)")

# 2) 정제 규칙 10종
df, applied = raw.copy(), []
def rule(name, n):
    applied.append((name, int(n)))

rule("R1 중복 보고 제거", df.duplicated().sum()); df = df.drop_duplicates().reset_index(drop=True)
bad_id = ~df["team"].str.match(r"^MT\d$"); df["team"] = "MT" + df["team"].str.extract(r"(\d+)")[0]; rule("R2 정비반 ID 표기 통일", bad_id.sum())
smap = {"ready": "ready", "대기": "ready", "moving": "moving", "repairing": "repairing"}
s = df["status"].fillna("").str.strip().str.lower().map(smap); rule("R3 상태값 표준화", (s != df["status"]).sum()); df["status"] = s
t1 = pd.to_datetime(df["ts"], format="%Y-%m-%d %H:%M", errors="coerce")
t2 = pd.to_datetime(df["ts"], format="%Y%m%d %H%M", errors="coerce"); rule("R4 보고시각 형식 통일", t1.isna().sum()); df["ts"] = t1.fillna(t2)
lit = df["fuel_unit"] == "L"; df.loc[lit, "fuel"] = df.loc[lit, "fuel"] / 4; df["fuel_unit"] = "%"; rule("R5 연료 단위 통일(L -> %)", lit.sum())
bad = ~df["crew"].between(1, 8) & df["crew"].notna(); df.loc[bad, "crew"] = np.nan; rule("R6 불가능 인원값 결측 처리", bad.sum())
bad = df["diag_kit"] < 0; df.loc[bad, "diag_kit"] = np.nan; rule("R7 음수 공구 수량 결측 처리", bad.sum())
bad = ~df["x"].between(0, 60); df.loc[bad, ["x", "y"]] = np.nan; df = df.sort_values(["team", "ts"])
df[["x", "y"]] = df.groupby("team")[["x", "y"]].ffill().bfill(); rule("R8 좌표 이상값 직전 위치 보간", bad.sum())
miss = df[["crew", "fuel", "diag_kit"]].isna().sum().sum()
df[["crew", "fuel", "diag_kit"]] = KNNImputer(n_neighbors=5).fit_transform(df[["crew", "fuel", "diag_kit", "lift_kit"]])[:, :3]
df["crew"] = df["crew"].round().astype(int); df["diag_kit"] = df["diag_kit"].round().astype(int); rule("R9 결측치 KNN 보정", miss)
q1, q3 = df["lift_kit"].quantile([0.25, 0.75]); out = df["lift_kit"] > q3 + 1.5 * (q3 - q1)
df["outlier_flag"] = out & ~df["emergency_issue"]; rule("R10 예외(긴급 불출) 보존·일반 이상값 표시", (out & df["emergency_issue"]).sum())
df["status"] = df.groupby("team")["status"].transform(lambda x: x.ffill().fillna("ready"))
df["dispatchable"] = (df["status"] == "ready") & (df["crew"] >= 3) & (df["fuel"] >= 30) & (df["diag_kit"] >= 1)
print("\n[적용 규칙과 처리 건수]")
for nme, n in applied:
    print(f" {nme:28s} {n:4d}건")

# 3) 정합성 검증
checks = {"중복 0건": not df.duplicated(["team", "ts"]).any(), "ID 형식": df["team"].str.match(r"^MT[1-6]$").all(),
          "상태값 허용 범위": df["status"].isin(["ready", "moving", "repairing"]).all(), "시각 변환": df["ts"].notna().all(),
          "인원 1~8명": df["crew"].between(1, 8).all(), "연료 0~100%": df["fuel"].between(0, 100).all(),
          "공구 수량 0 이상": (df["diag_kit"] >= 0).all(), "좌표 범위": df["x"].between(0, 60).all(),
          "결측 0건": df[["crew", "fuel", "diag_kit", "x", "y", "status"]].notna().all().all(),
          f"긴급 불출 {len(k)}건 보존": int(df["emergency_issue"].sum()) == len(k)}
rate = np.mean(list(checks.values()))
print(f"\n정합성 검증 {sum(checks.values())}/{len(checks)} 통과, 통과율 {rate:.0%} (목표 100%: {'달성' if rate == 1 else '미달'})")
print(f"정제 후 {len(df)}건, 출동 가능 보고 비율 {df['dispatchable'].mean():.1%}")

# 4) 정제 전후 기술통계 비교
num = lambda x: pd.to_numeric(x, errors="coerce")
cmp = pd.DataFrame({"정제 전 평균": [num(raw[c]).mean() for c in ["crew", "fuel", "diag_kit", "x"]],
                    "정제 후 평균": [df[c].mean() for c in ["crew", "fuel", "diag_kit", "x"]],
                    "정제 전 최대": [num(raw[c]).max() for c in ["crew", "fuel", "diag_kit", "x"]],
                    "정제 후 최대": [df[c].max() for c in ["crew", "fuel", "diag_kit", "x"]]}, index=["인원", "연료%", "진단키트", "좌표X"]).round(1)
print("\n[정제 전후 기술통계]"); print(cmp.to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(num(raw["fuel"]).dropna(), bins=30, alpha=0.6, label="raw"); ax[0].hist(df["fuel"], bins=30, alpha=0.6, label="cleaned")
ax[0].set_title("Fuel level before/after unit fix"); ax[0].legend()
ax[1].bar([n.split()[0] for n, _ in applied], [n for _, n in applied]); ax[1].set_title("Records changed by rule")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
원천 로그 372건 (중복 12건, 오류 주입 150건 포함)

[적용 규칙과 처리 건수]
 R1 중복 보고 제거                    12건
 R2 정비반 ID 표기 통일                15건
 R3 상태값 표준화                     30건
 R4 보고시각 형식 통일                  15건
 R5 연료 단위 통일(L -> %)            15건
 R6 불가능 인원값 결측 처리               15건
 R7 음수 공구 수량 결측 처리              15건
 R8 좌표 이상값 직전 위치 보간             15건
 R9 결측치 KNN 보정                  60건
 R10 예외(긴급 불출) 보존·일반 이상값 표시     15건

정합성 검증 10/10 통과, 통과율 100% (목표 100%: 달성)
정제 후 360건, 출동 가능 보고 비율 60.3%

[정제 전후 기술통계]
      정제 전 평균  정제 후 평균  정제 전 최대  정제 후 최대
인원        4.2      4.0     25.0      5.0
연료%      74.1     66.9    366.0    100.0
진단키트      1.4      1.5      2.0      2.0
좌표X      71.8     30.1    999.0     59.8
```
</details>
